In [ ]:
import scipy.stats as stats
import scipy
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

## Exercise 1 

Task is to generate multivariate normal distribution using transformation method:

**Obtain normal distribution**

First we need get $k=2$ normal distributions:
$$\begin{align*} 
Z &\sim (\mathcal{N}_1, \mathcal{N}_2) & \text{where } \mathcal{N}_i \sim \mathcal{N}(0, 1) 
\end{align*}$$
Then using cholesky decomposition we obtain lower ($L$) matrix, and use transformation: 
$$X = L \cdot Z $$

To make it efficient we need to vectorize results

**Plot results with isoline**

In [ ]:
# helpers
def multivariate_normal(n, V):
    # for clarification 
    N = n 
    D = V.shape[0]
    L = np.linalg.cholesky(V)
    probe = stats.distributions.norm().rvs([D, N]) # (D, N) probe - Z matrix
    result = L @ probe
    return result

def plot_multivariate_2d(x, y):
    # --- Plot  ---
    plt.figure(figsize=(10, 8))

    # 1. Scatter plot (czarne punkty, mały rozmiar, lekko przezroczyste)
    plt.scatter(x, y, s=2, color='black', alpha=0.6, label='Próbki')

    # 2. Izolinie PDF (KDE plot za pomocą seaborn)
    sns.kdeplot(x=x, y=y, levels=8, color="blue", linewidths=1.2)

    # 3. Dodanie czerwonej linii (opcjonalnie, np. linii regresji y na x)
    # Współczynnik nachylenia = Cov(x,y) / Var(x) = 2 / 1 = 2
    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = 2 * x_line 
    plt.plot(x_line, y_line, color='red', linewidth=1, label='Linia trendu')

    plt.title('Distribution with isolines (KDE)')
    plt.xlabel('X1')
    plt.ylabel('X2')
    plt.legend()
    plt.show()

In [ ]:
# params 
## example one 
V = np.array([
    [1, 2],
     [2, 5]
    ])

## results 
probe_1000 = multivariate_normal(n=1000, V=V)
plot_multivariate_2d(probe_1000[0, :], probe_1000[1, :])
probe_1_000_00 = multivariate_normal(n=1_000_00, V=V)
plot_multivariate_2d(probe_1_000_00[0, :], probe_1_000_00[1, :])


## Exercise one 

Plot ball $U$ and circle sphere $S$ by using rejection sampling 

$$U^{d} = \{x \in \mathbb{R}^d: |x|=1\}$$
$$S^{d-1} = \{x \in \mathbb{R}^d: |x|=1\}$$

In [ ]:
# rejection sampling. to ball 


def disk_unif_rvs(n: int, disk_dim: int):
    # rejection sampling
    probe = np.zeros([n, disk_dim], dtype=np.float32)
    idx = 0
    # TODO - apply here right condition
    while idx < n:
        ## obtain probe from uniform distribution 
        rvs = stats.distributions.uniform(-1, 2).rvs([n-idx, disk_dim])
        ## check which idx norm <= 1 (are in ball) along 
        accept_filt = np.apply_along_axis(
            lambda row: np.linalg.norm(row) <= 1, 
            axis=1, 
            arr=rvs)
        rvs = rvs[accept_filt]
        probe[idx: idx + rvs.shape[0]] = rvs
        idx += rvs.shape[0]
    
    return probe
    
def sphere_unif_rvs(n: int, spere_dim: int):
    # get disk sample
    disk_sample = disk_unif_rvs(n=n, disk_dim=spere_dim + 1)
    # move elements to the sphere
    sphere_sample = np.apply_along_axis(
        lambda row: row / np.linalg.norm(row),
        axis=1,
        arr=disk_sample
    )
    return sphere_sample

In [ ]:
# get sample
disk_sample = disk_unif_rvs(10_000, 2)
sphere_sample = sphere_unif_rvs(10_000, 1)

# plot
plt.scatter(disk_sample[:, 0], disk_sample[:, 1], color='blue', alpha=0.4, s=2)
plt.scatter(sphere_sample[:, 0], sphere_sample[:, 1], color='red', alpha=0.4, s=2)
# plt.xlim((-1, 1))
# plt.ylim((-1, 1))

Jeśli mamy taki algorytm (kiswiabue z birnkabeig u kuli) 
X = (X_1, X_2, X_3) \sim Unif(S^2)

and find marginal for X_1, X_2 

Marignalizacja w przypadku symulacji to jest po prostu ignorowanie wszystkich pozostałych zmiennych i branie po prostu rozkładu po wartości z tej zmiennej którą jest marginalizowana (bo normlanie mamy rozkład np (U, V), i tutaj po prostu bierzemy U więc wystarczy że weźmiemy histogram z U )